In [ ]:
import pandas as pd
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import time
import re
import warnings
warnings.filterwarnings('ignore')

performance = pd.DataFrame(columns=['Name', 'Accuracy', 'Precision', 'Sensitivity', 'F1 Score', 'MCC', 'Markedness', "Youden's J", 'FMI', 'Time'])

df = pd.read_csv("StealthPhisher2025.csv")
LABEL = df.iloc[:,-1:].columns[0]

colsAll = df.select_dtypes(include=['float64','int64']).columns
colsAll.append(pd.Index([LABEL]))
df = pd.DataFrame(df[colsAll]).copy()

print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Embedding, Flatten, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Assuming `df` is your DataFrame and `LABEL` is the target column
X = df.drop(columns=[LABEL]).values
y = df[LABEL].values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the Wide and Deep Neural Network
def build_wide_and_deep(input_dim):
    # Input layer
    inputs = Input(shape=(input_dim,))
    wide = Dense(1)(inputs)
    deep = Dense(128, activation='relu')(inputs)
    deep = Dense(64, activation='relu')(deep)
    deep = Dense(32, activation='relu')(deep)
    combined = concatenate([wide, deep])
    outputs = Dense(1, activation='sigmoid')(combined)    
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_wide_and_deep(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()
y_pred = (model.predict(X_test) > 0.5).astype("int32")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

total_time = end_time - start_time

newRow = {
    'Name': 'Wide and Deep Neural Network',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("performance_metrics.csv", index=False, mode='a')
performance

In [ ]:
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)
# Define Sparse Neural Network block
class SparseLayer(tf.keras.layers.Layer):
    def __init__(self, units, sparsity_coefficient=0.001, **kwargs):
        super(SparseLayer, self).__init__(**kwargs)
        self.units = units
        self.sparsity_coefficient = sparsity_coefficient

    def build(self, input_shape):
        # Define weights and biases with L1 regularization
        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer="random_normal",
                                 regularizer=tf.keras.regularizers.L1(self.sparsity_coefficient),
                                 trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

# Define Sparse Neural Network Model
def build_sparse_nn(input_dim, sparsity_coefficient=0.001):
    inputs = Input(shape=(input_dim,))
    
    # Sparse layers with L1 regularization
    x = SparseLayer(128, sparsity_coefficient=sparsity_coefficient)(inputs)
    x = SparseLayer(64, sparsity_coefficient=sparsity_coefficient)(x)
    
    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the Sparse Neural Network model
model = build_sparse_nn(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Sparse Neural Network (SNN)',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("performance_metrics.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.units = units
        self.attention_dense = Dense(units, activation='tanh')
        self.attention_weights = Dense(1, activation='softmax')
    
    def call(self, inputs):
        # Compute attention scores
        attention_scores = self.attention_dense(inputs)
        attention_scores = self.attention_weights(attention_scores)
        # Apply attention scores to input features
        weighted_inputs = Multiply()([inputs, attention_scores])
        return weighted_inputs

# Define Attention-based MLP Model
def build_attention_mlp(input_dim, attention_units=64, dense_units=[128, 64, 32]):
    inputs = Input(shape=(input_dim,))
    
    # Attention layer
    attention_layer = AttentionLayer(units=attention_units)
    attention_output = attention_layer(inputs)
    
    # Dense layers after attention
    x = Dense(dense_units[0], activation='relu')(attention_output)
    x = Dense(dense_units[1], activation='relu')(x)
    x = Dense(dense_units[2], activation='relu')(x)
    
    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the Attention-based MLP model
model = build_attention_mlp(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Attention-based MLP Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("performance_metrics.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Layer, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define NODE Block
class NODEBlock(Layer):
    def __init__(self, num_trees, tree_depth, output_dim, **kwargs):
        super(NODEBlock, self).__init__(**kwargs)
        self.num_trees = num_trees
        self.tree_depth = tree_depth
        self.output_dim = output_dim

        # Initialize layers for the "oblivious" decision trees
        self.feature_selectors = [Dense(1) for _ in range(tree_depth * num_trees)]
        self.tree_weights = Dense(output_dim)

    def call(self, inputs):
        # Compute tree outputs
        tree_outputs = []
        for tree in range(self.num_trees):
            tree_output = inputs
            for depth in range(self.tree_depth):
                feature_selector = self.feature_selectors[tree * self.tree_depth + depth]
                tree_output = tf.sigmoid(feature_selector(tree_output))
            tree_outputs.append(tree_output)

        # Concatenate and combine tree outputs
        combined_trees = tf.concat(tree_outputs, axis=1)
        node_output = self.tree_weights(combined_trees)
        return node_output

# Define the NODE-based model
def build_node_model(input_dim, num_trees=5, tree_depth=3, output_dim=64):
    inputs = Input(shape=(input_dim,))
    
    # NODE Block
    node_block = NODEBlock(num_trees=num_trees, tree_depth=tree_depth, output_dim=output_dim)
    node_output = node_block(inputs)
    
    # Additional dense layers for hybrid model
    x = Dense(128, activation='relu')(node_output)
    x = Dropout(0.3)(x)  # Dropout for regularization
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the NODE-based model
model = build_node_model(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=16, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Improved NODE-based Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("performance_metrics.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define the Cross Layer for DCN
class CrossLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(CrossLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[1][-1], 1), initializer="random_normal", trainable=True)
        self.b = self.add_weight(shape=(input_shape[1][-1],), initializer="zeros", trainable=True)

    def call(self, inputs):
        x0, x = inputs
        cross_term = tf.matmul(x, self.w) * x0 + self.b
        return cross_term + x

# Define the Deep & Cross Network
def build_dcn(input_dim, num_cross_layers=3, num_deep_layers=3, deep_units=64, dropout_rate=0.3):
    inputs = Input(shape=(input_dim,))
    
    # Cross Network
    x0 = inputs
    x = x0
    for _ in range(num_cross_layers):
        x = CrossLayer()([x0, x])
    
    # Deep Network
    d = inputs
    for _ in range(num_deep_layers):
        d = Dense(deep_units, activation='relu')(d)
        d = BatchNormalization()(d)  # Batch Normalization for stability
        d = Dropout(dropout_rate)(d)  # Dropout for regularization
    
    # Combine Cross and Deep Network outputs
    combined = Concatenate()([x, d])
    outputs = Dense(1, activation='sigmoid')(combined)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Learning rate scheduler
def lr_schedule(epoch):
    initial_lr = 0.01
    return initial_lr * (0.5 ** (epoch // 3))

# Initialize and compile the DCN model
model = build_dcn(input_dim=X_train.shape[1], dropout_rate=0.3)
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
callbacks = [LearningRateScheduler(lr_schedule)]
history = model.fit(X_train, y_train, epochs=10, batch_size=16, validation_split=0.2, verbose=1, callbacks=callbacks)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Deep & Cross Network (DCN)',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

newRow

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define Feed-Forward Neural Network Model
def build_ffnn(input_dim):
    inputs = Input(shape=(input_dim,))
    
    # Dense layers for FFNN
    x = Dense(128, activation='relu')(inputs)
    x = Dense(64, activation='relu')(x)
    x = Dense(32, activation='relu')(x)
    
    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the Feed-Forward Neural Network model
model = build_ffnn(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Feed-Forward Neural Network',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

newRow

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.layers import Dense, Flatten, Input, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Pad data to match the 32x32 requirement for MobileNetV3
X_train_padded = np.pad(X_train, ((0, 0), (0, 32 * 32 - X_train.shape[1])), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test, ((0, 0), (0, 32 * 32 - X_test.shape[1])), 'constant').reshape(-1, 32, 32, 1)

# Define MobileNetV3 model for feature extraction
base_model = MobileNetV3Small(input_shape=(32, 32, 1), include_top=False, weights=None)
base_model.trainable = True  # Allow fine-tuning

# Build hybrid model
inputs = Input(shape=(32, 32, 1))
x = base_model(inputs)
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)  # Batch normalization
x = Dropout(0.3)(x)  # Dropout for regularization
x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)

# Learning rate scheduler
def lr_schedule(epoch):
    initial_lr = 0.001
    return initial_lr * (0.5 ** (epoch // 3))

# Compile model
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
callbacks = [LearningRateScheduler(lr_schedule)]
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1, callbacks=callbacks)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'MobileNetV3 Hybrid Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

newRow

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

class TabNetBlock(tf.keras.layers.Layer):
    def __init__(self, feature_dim, output_dim, num_steps, relaxation_factor, sparsity_coefficient, **kwargs):
        super(TabNetBlock, self).__init__(**kwargs)
        self.feature_dim = feature_dim
        self.output_dim = output_dim
        self.num_steps = num_steps
        self.relaxation_factor = relaxation_factor
        self.sparsity_coefficient = sparsity_coefficient

        # Define layers for feature transformer and attention mechanism
        self.shared_transform = Dense(self.feature_dim, activation='relu')
        self.attentive_transform = Dense(self.feature_dim, activation='sigmoid')
        self.step_output = Dense(self.output_dim, activation='relu')

    def call(self, inputs):
        # Initialize prior scale with the same feature dimension as inputs
        prior_scale = tf.ones((tf.shape(inputs)[0], tf.shape(inputs)[-1]))  # Dynamic adjustment
        aggregated_output = 0.0
        sparse_loss = 0.0

        for step in range(self.num_steps):
            # Feature selection using an attention mechanism
            mask_values = self.attentive_transform(prior_scale)
            masked_inputs = tf.multiply(inputs, mask_values)

            # Feature transformer
            transformed_inputs = self.shared_transform(masked_inputs)
            step_output = self.step_output(transformed_inputs)
            aggregated_output += step_output

            # Sparse loss
            sparse_loss += tf.reduce_mean(mask_values * (1 - mask_values))

            # Update prior scale based on relaxation factor
            prior_scale = self.relaxation_factor * prior_scale - mask_values

        # Add sparse loss as a custom loss for regularization
        self.add_loss(sparse_loss * self.sparsity_coefficient)
        return aggregated_output  # Only return the final output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.output_dim)

# Define the TabNet Hybrid Model
def build_tabnet_hybrid(input_dim, feature_dim=64, output_dim=32, num_steps=3, relaxation_factor=1.5, sparsity_coefficient=0.0001):
    inputs = Input(shape=(input_dim,))
    
    # TabNet block
    tabnet_block = TabNetBlock(feature_dim=input_dim, output_dim=output_dim, num_steps=num_steps, 
                               relaxation_factor=relaxation_factor, sparsity_coefficient=sparsity_coefficient)
    tabnet_output = tabnet_block(inputs)  # Only one output now

    # Additional dense layers for hybrid model
    x = Dense(128, activation='relu')(tabnet_output)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the TabNet Hybrid model
model = build_tabnet_hybrid(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'TabNet Hybrid Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

newRow

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Conv2D, DepthwiseConv2D, BatchNormalization, ReLU, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Pad data to match the 32x32 input shape
X_train_padded = np.pad(X_train, ((0, 0), (0, max(0, 32 * 32 - X_train.shape[1]))), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test, ((0, 0), (0, max(0, 32 * 32 - X_test.shape[1]))), 'constant').reshape(-1, 32, 32, 1)

def micronet_block(x, out_channels, stride=1):
    x = DepthwiseConv2D(kernel_size=3, strides=stride, padding='same', depth_multiplier=1)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(out_channels, kernel_size=1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

# Model definition
inputs = Input(shape=(32, 32, 1))
x = Conv2D(16, kernel_size=3, strides=2, padding='same')(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)
x = micronet_block(x, 32, stride=2)
x = micronet_block(x, 64, stride=2)
x = micronet_block(x, 128, stride=2)
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)  # Weight decay added
x = Dropout(0.5)(x)  # Increased dropout
x = Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)  # Reduced neurons
x = Dropout(0.5)(x)  # Increased dropout
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

# Compile with weight decay
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Learning rate schedule with faster decay
def lr_schedule(epoch):
    return 0.01 * (0.4 ** (epoch // 3))  # Increased decay rate

callbacks = [LearningRateScheduler(lr_schedule)]

# Train the model
start_time = time.time()
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=16, validation_split=0.2, verbose=1, callbacks=callbacks)
end_time = time.time()

# Predictions
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Total time taken
total_time = end_time - start_time

# Performance dictionary
newRow = {
    'Name': 'MicroNet Hybrid Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

newRow


In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Input, Conv2D, MaxPooling2D, BatchNormalization, ReLU, Add, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Pad data to match the 32x32 input shape
X_train_padded = np.pad(X_train, ((0, 0), (0, max(0, 32 * 32 - X_train.shape[1]))), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test, ((0, 0), (0, max(0, 32 * 32 - X_test.shape[1]))), 'constant').reshape(-1, 32, 32, 1)

# Define a modified ShuffleNetV2 block
def shufflenet_v2_block(x, out_channels, downsample=False):
    shortcut = x
    strides = 2 if downsample else 1

    # Depthwise convolution branch
    x = Conv2D(out_channels // 2, kernel_size=1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(out_channels // 2, kernel_size=3, strides=strides, padding='same', groups=out_channels // 2)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(out_channels // 2, kernel_size=1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)

    # Shortcut path
    if downsample:
        shortcut = Conv2D(out_channels // 2, kernel_size=3, strides=2, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Concatenate paths
    x = Add()([x, shortcut])
    return x

# Assemble a modified ShuffleNetV2 backbone
inputs = Input(shape=(32, 32, 1))
x = Conv2D(16, kernel_size=3, strides=2, padding='same')(inputs)  # Reduced initial filters
x = BatchNormalization()(x)
x = ReLU()(x)

# Stack several ShuffleNetV2 blocks with reduced complexity
x = shufflenet_v2_block(x, 32, downsample=True)  # Reduced output channels
x = shufflenet_v2_block(x, 64, downsample=True)
x = shufflenet_v2_block(x, 128, downsample=True)

x = GlobalAveragePooling2D()(x)

base_model = Model(inputs, x)

# Build the final model with Dropout
x = Dense(64, activation='relu')(base_model.output)  # Reduced neurons
x = Dropout(0.5)(x)  # Dropout to regularize
x = Dense(32, activation='relu')(x)  # Further reduced neurons
x = Dropout(0.5)(x)  # Dropout to regularize
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)  # Fowlkes-Mallows Index

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'ShuffleNetV2 Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

newRow

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Input, Conv2D, BatchNormalization, ReLU, Add, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Add noise to the input data
noise_factor = 0.05
X_train_noisy = X_train + noise_factor * np.random.normal(size=X_train.shape)
X_test_noisy = X_test + noise_factor * np.random.normal(size=X_test.shape)

# Pad data to match the 32x32 input shape
X_train_padded = np.pad(X_train_noisy, ((0, 0), (0, max(0, 32 * 32 - X_train_noisy.shape[1]))), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test_noisy, ((0, 0), (0, max(0, 32 * 32 - X_test_noisy.shape[1]))), 'constant').reshape(-1, 32, 32, 1)

# Define a RegNet block with regularization
def regnet_block(x, out_channels, stride=1):
    """RegNet Block with bottleneck structure and regularization."""
    shortcut = x

    # 1x1 Convolution (Bottleneck Layer)
    x = Conv2D(out_channels, kernel_size=1, strides=1, padding='same', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # 3x3 Convolution (Main Convolution)
    x = Conv2D(out_channels, kernel_size=3, strides=stride, padding='same', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # 1x1 Convolution (Expansion Layer)
    x = Conv2D(out_channels, kernel_size=1, strides=1, padding='same', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)

    # Shortcut connection with a downsample layer if needed
    if stride != 1 or shortcut.shape[-1] != out_channels:
        shortcut = Conv2D(out_channels, kernel_size=1, strides=stride, padding='same', kernel_regularizer=l2(0.01))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add shortcut and main path
    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

# Define the RegNet-inspired backbone model
inputs = Input(shape=(32, 32, 1))
x = Conv2D(16, kernel_size=3, strides=2, padding='same', kernel_regularizer=l2(0.01))(inputs)  # Fewer filters
x = BatchNormalization()(x)
x = ReLU()(x)

# Stack several RegNet blocks
x = regnet_block(x, 32, stride=2)  # Fewer channels
x = regnet_block(x, 64, stride=2)
x = regnet_block(x, 128, stride=2)

x = GlobalAveragePooling2D()(x)

base_model = Model(inputs, x)

# Build the final model
x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(base_model.output)
x = Dropout(0.5)(x)  # Added dropout
x = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(x)
x = Dropout(0.5)(x)  # Added dropout
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)

# Compile the model with a suboptimal learning rate
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'RegNet Hybrid Model with Noise and Regularization',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

print(newRow)